# 🎯 EfficientUNet++ Background Removal — Complete Training

**Architecture:** EfficientUNet++ (EfficientNet-B4 Encoder + U-Net++ Decoder + CBAM + Boundary Head)

**Dataset:** P3M-10K (Portrait Matting)

**Target:** IoU > 0.95

| Component | Details |
|---|---|
| Encoder | EfficientNet-B4 style MBConv (~5M params) |
| Decoder | U-Net++ nested skips + CBAM attention |
| Loss | BCE + Dice + Focal + IoU + Tversky + Lovász + Boundary |
| Optimizer | AdamW (lr=2e-4, wd=2e-4) |
| Schedule | Warmup + CosineAnnealingWarmRestarts |
| Training | AMP + Gradient Accumulation + EMA + Mixup |
| Progressive | 192→256→320 input size across 70 epochs |
| Export | Single merged ONNX (no external data) |

---

## CELL 1 — Install + Setup

In [ ]:
# ================================================================
# CELL 1 — Install + Setup
# ================================================================
!pip install -q kagglehub albumentations>=1.3.0 onnx>=1.14.0 onnxruntime>=1.15.0

import os, sys, gc, time, random, math, shutil, warnings, glob
from pathlib import Path
from collections import OrderedDict

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')

# ── Seeds ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# ── Device ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('WARNING: No GPU! Training will be very slow.')

# ── CONFIG ──
CONFIG = {
    'seed': SEED,
    'epochs': 70,
    'warmup_epochs': 5,
    'patience': 20,
    'batch_size': 4,
    'accum_steps': 8,
    'num_workers': 2,
    'pin_memory': torch.cuda.is_available(),
    'lr': 2e-4,
    'weight_decay': 2e-4,
    'T_0': 10,
    'T_mult': 2,
    'eta_min': 1e-7,
    'ema_decay': 0.999,
    'label_smooth': 0.95,
    'mixup_prob': 0.3,
    'mixup_alpha': 0.4,
    'grad_clip': 1.0,
    # Progressive sizes
    'phase1_size': 192,
    'phase2_size': 256,
    'phase3_size': 320,
    'phase2_epoch': 20,
    'phase3_epoch': 50,
    'phase1_lr': 2e-4,
    'phase2_lr': 1e-4,
    'phase3_lr': 5e-5,
    # Paths
    'local_ckpt_dir': '/content/checkpoints',
    'drive_dir': '/content/drive/MyDrive/bg_removal_ai',
}

os.makedirs(CONFIG['local_ckpt_dir'], exist_ok=True)
print('\n\u2713 Setup complete')
print(f'Config: {len(CONFIG)} hyperparameters set')

## CELL 2 — Google Drive

In [ ]:
# ================================================================
# CELL 2 — Google Drive Mount
# ================================================================
DRIVE_MOUNTED = False
DRIVE_DIR = CONFIG['drive_dir']

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_DIR}/exports', exist_ok=True)
    DRIVE_MOUNTED = True
    print(f'Drive mounted at {DRIVE_DIR}')
    print(f'  Checkpoints: {DRIVE_DIR}/checkpoints/')
    print(f'  Exports:     {DRIVE_DIR}/exports/')
except Exception as e:
    print(f'Drive mount failed: {e}')
    print('Checkpoints will be saved locally only.')
    DRIVE_MOUNTED = False


## CELL 3 — Dataset Download

In [ ]:
# ================================================================
# CELL 3 — Dataset Download + Discovery
# ================================================================
import kagglehub

print('Downloading P3M-10K dataset...')
dataset_path = kagglehub.dataset_download('rahulbhalley/p3m-10k')
print(f'Dataset path: {dataset_path}')

# ── Auto-discover image and mask directories ──
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IMAGE_KEYWORDS = {'image', 'blurred', 'original', 'img', 'photo'}
MASK_KEYWORDS = {'mask', 'alpha', 'matte', 'trimap', 'segment'}

def discover_dirs(root):
    """Walk root and find directories with >50 image files."""
    image_dirs = []
    mask_dirs = []
    for dirpath, dirnames, filenames in os.walk(root):
        img_count = sum(1 for f in filenames if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS)
        if img_count < 50:
            continue
        dirname_lower = dirpath.lower()
        is_mask = any(kw in dirname_lower for kw in MASK_KEYWORDS)
        is_image = any(kw in dirname_lower for kw in IMAGE_KEYWORDS)
        if is_mask:
            mask_dirs.append((dirpath, img_count))
        elif is_image:
            image_dirs.append((dirpath, img_count))
        else:
            image_dirs.append((dirpath, img_count))
    return image_dirs, mask_dirs

img_dirs, msk_dirs = discover_dirs(dataset_path)

print(f'\nImage directories found ({len(img_dirs)}):')
for d, c in img_dirs:
    print(f'  {d} ({c} files)')

print(f'\nMask directories found ({len(msk_dirs)}):')
for d, c in msk_dirs:
    print(f'  {d} ({c} files)')

# Select largest image and mask dirs
if not img_dirs:
    raise RuntimeError('No image directories found! Check dataset structure.')
if not msk_dirs:
    raise RuntimeError('No mask directories found! Check dataset structure.')

IMAGE_DIR = max(img_dirs, key=lambda x: x[1])[0]
MASK_DIR = max(msk_dirs, key=lambda x: x[1])[0]

print(f'\n\u2713 Using:')
print(f'  Images: {IMAGE_DIR}')
print(f'  Masks:  {MASK_DIR}')

## CELL 4 — Dataset + Loaders

In [ ]:
# ================================================================
# CELL 4 — Dataset Class + Augmentations + Loaders
# ================================================================

def make_transforms(size, training=True):
    """Build albumentations pipeline."""
    if training:
        return A.Compose([
            A.Resize(size, size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.1),
            A.ShiftScaleRotate(
                shift_limit=0.1, scale_limit=0.25, rotate_limit=20,
                border_mode=cv2.BORDER_REFLECT, p=0.7),
            A.RandomBrightnessContrast(0.3, 0.3, p=0.6),
            A.HueSaturationValue(20, 30, 20, p=0.5),
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),
            A.GaussianBlur(blur_limit=(3, 7), p=0.3),
            A.GaussNoise(var_limit=(10, 50), p=0.2),
            A.CoarseDropout(
                max_holes=6, max_height=48, max_width=48,
                fill_value=0, p=0.3),
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(size, size),
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])


class SegDataset(Dataset):
    """Segmentation dataset with auto-pairing by filename stem."""

    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        try:
            image = np.array(Image.open(img_path).convert('RGB'))
            mask = np.array(Image.open(mask_path).convert('L'))
            mask = (mask > 127).astype(np.float32)
        except Exception:
            image = np.zeros((320, 320, 3), dtype=np.uint8)
            mask = np.zeros((320, 320), dtype=np.float32)

        if self.transform:
            t = self.transform(image=image, mask=mask)
            image = t['image']
            mask = t['mask']

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        return {'image': image, 'mask': mask}


def build_pairs(image_dir, mask_dir):
    """Match images to masks by filename stem."""
    img_files = {}
    for f in os.listdir(image_dir):
        ext = os.path.splitext(f)[1].lower()
        if ext in IMAGE_EXTENSIONS:
            stem = os.path.splitext(f)[0]
            img_files[stem] = os.path.join(image_dir, f)

    mask_files = {}
    for f in os.listdir(mask_dir):
        ext = os.path.splitext(f)[1].lower()
        if ext in IMAGE_EXTENSIONS:
            stem = os.path.splitext(f)[0]
            mask_files[stem] = os.path.join(mask_dir, f)

    pairs = []
    for stem in sorted(img_files.keys()):
        if stem in mask_files:
            pairs.append((img_files[stem], mask_files[stem]))

    print(f'  Images: {len(img_files)}, Masks: {len(mask_files)}, Paired: {len(pairs)}')
    return pairs


# ── Build pairs ──
print('Building image-mask pairs...')
all_pairs = build_pairs(IMAGE_DIR, MASK_DIR)
assert len(all_pairs) > 100, f'Only {len(all_pairs)} pairs found!'

# ── Split 80/10/10 ──
rng = np.random.RandomState(SEED)
indices = rng.permutation(len(all_pairs))
n = len(all_pairs)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_pairs = [all_pairs[i] for i in indices[:n_train]]
val_pairs = [all_pairs[i] for i in indices[n_train:n_train + n_val]]
test_pairs = [all_pairs[i] for i in indices[n_train + n_val:]]

print(f'\nSplits: Train={len(train_pairs)}, Val={len(val_pairs)}, Test={len(test_pairs)}')


def build_loaders(size):
    """Build train/val/test dataloaders at given input size."""
    train_ds = SegDataset(train_pairs, make_transforms(size, training=True))
    val_ds = SegDataset(val_pairs, make_transforms(size, training=False))
    test_ds = SegDataset(test_pairs, make_transforms(size, training=False))

    kw = dict(
        batch_size=CONFIG['batch_size'],
        num_workers=CONFIG['num_workers'],
        pin_memory=CONFIG['pin_memory'],
        prefetch_factor=2,
        persistent_workers=True,
    )
    train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, **kw)
    val_loader = DataLoader(val_ds, shuffle=False, **kw)
    test_loader = DataLoader(test_ds, shuffle=False, **kw)
    print(f'  Loaders built at size={size}: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)} batches')
    return train_loader, val_loader, test_loader


# ── Build initial loaders at phase 1 size ──
train_loader, val_loader, test_loader = build_loaders(CONFIG['phase1_size'])
print('\n\u2713 Dataset and loaders ready')

## CELL 5 — Model Architecture

In [ ]:
# ================================================================
# CELL 5 — Model Architecture (EfficientUNet++)
# ================================================================

class HSwish(nn.Module):
    def forward(self, x):
        return x * F.relu6(x + 3.0, inplace=False) / 6.0


class SEBlock(nn.Module):
    def __init__(self, ch, r=4):
        super().__init__()
        mid = max(ch // r, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(ch, mid, 1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, ch, 1, bias=True),
        )

    def forward(self, x):
        s = self.pool(x)
        s = self.fc(s)
        s = F.relu6(s + 3.0, inplace=False) / 6.0  # h-sigmoid
        return x * s


class MBConv(nn.Module):
    """Mobile Inverted Bottleneck: expand -> depthwise -> SE -> project."""
    def __init__(self, inp, oup, stride, expand, kernel=3, use_se=False):
        super().__init__()
        self.use_residual = (stride == 1 and inp == oup)
        mid = int(round(inp * expand))
        layers = []
        if expand != 1:
            layers += [
                nn.Conv2d(inp, mid, 1, bias=False),
                nn.BatchNorm2d(mid),
                HSwish(),
            ]
        layers += [
            nn.Conv2d(mid, mid, kernel, stride=stride,
                      padding=kernel // 2, groups=mid, bias=False),
            nn.BatchNorm2d(mid),
            HSwish(),
        ]
        if use_se:
            layers.append(SEBlock(mid))
        layers += [
            nn.Conv2d(mid, oup, 1, bias=False),
            nn.BatchNorm2d(oup),
        ]
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        out = self.block(x)
        return out + x if self.use_residual else out


class ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 4)
        self.mlp = nn.Sequential(
            nn.Linear(ch, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, ch, bias=False),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        gap = x.mean(dim=(2, 3))
        gmp = x.amax(dim=(2, 3))
        gate = torch.sigmoid(self.mlp(gap) + self.mlp(gmp))
        return x * gate.view(B, C, 1, 1)


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx = x.amax(dim=1, keepdim=True)
        gate = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * gate


class CBAM(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.ca = ChannelAttention(ch, r)
        self.sa = SpatialAttention()

    def forward(self, x):
        return self.sa(self.ca(x))


class DecoderNode(nn.Module):
    """U-Net++ style decoder node: upsample + concat skips + conv + CBAM."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        total_in = in_ch + skip_ch
        self.conv = nn.Sequential(
            nn.Conv2d(total_in, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )
        self.cbam = CBAM(out_ch)
        self.drop = nn.Dropout2d(0.1)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        x = self.cbam(x)
        x = self.drop(x)
        return x


class BoundaryHead(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        mid = max(in_ch // 2, 8)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, mid, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, 1, 1),
        )

    def forward(self, x):
        return self.conv(x)


class EfficientUNetPP(nn.Module):
    """
    EfficientNet-B4 encoder + U-Net++ decoder + CBAM + Boundary head.
    All outputs are RAW LOGITS (no sigmoid).
    """
    def __init__(self):
        super().__init__()
        # ── Encoder ──
        self.stem = nn.Sequential(
            nn.Conv2d(3, 48, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(48),
            HSwish(),
        )
        self.e1 = nn.Sequential(
            MBConv(48, 24, stride=2, expand=1, kernel=3),
            MBConv(24, 24, stride=1, expand=1, kernel=3),
        )
        self.e2 = nn.Sequential(
            MBConv(24, 32, stride=2, expand=6, kernel=3),
            MBConv(32, 32, stride=1, expand=6, kernel=3),
        )
        self.e3 = nn.Sequential(
            MBConv(32, 56, stride=2, expand=6, kernel=5, use_se=True),
            MBConv(56, 56, stride=1, expand=6, kernel=5, use_se=True),
            MBConv(56, 56, stride=1, expand=6, kernel=5, use_se=True),
        )
        self.e4 = nn.Sequential(
            MBConv(56, 112, stride=2, expand=6, kernel=3, use_se=True),
            MBConv(112, 112, stride=1, expand=6, kernel=3, use_se=True),
            MBConv(112, 112, stride=1, expand=6, kernel=3, use_se=True),
        )
        self.e5 = nn.Sequential(
            nn.Conv2d(112, 672, 1, bias=False),
            nn.BatchNorm2d(672),
            HSwish(),
        )

        # ── Decoder ──
        self.d1 = DecoderNode(672, 112, 256)
        self.d2 = DecoderNode(256, 56, 128)
        self.d3 = DecoderNode(128, 32, 64)
        self.d4 = DecoderNode(64, 24, 32)
        self.d5 = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.GELU(),
            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.GELU(),
        )

        # ── Output heads (RAW LOGITS, no sigmoid) ──
        self.out_head = nn.Conv2d(16, 1, 1)
        self.s2_head = nn.Conv2d(128, 1, 1)
        self.s3_head = nn.Conv2d(64, 1, 1)
        self.s4_head = nn.Conv2d(32, 1, 1)
        self.boundary_head = BoundaryHead(17)  # 16 + 1 (main logit)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        H, W = x.shape[2:]
        s = self.stem(x)
        e1 = self.e1(s)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)
        e5 = self.e5(e4)

        d1 = self.d1(e5, e4)
        d2 = self.d2(d1, e3)
        d3 = self.d3(d2, e2)
        d4 = self.d4(d3, e1)
        d5 = self.d5(F.interpolate(d4, size=(H, W), mode='bilinear', align_corners=False))

        main_logit = self.out_head(d5)

        if self.training:
            up = lambda t: F.interpolate(t, size=(H, W), mode='bilinear', align_corners=False)
            boundary_logit = self.boundary_head(torch.cat([d5, main_logit], dim=1))
            return {
                'main': main_logit,
                'boundary': boundary_logit,
                'deep': [
                    up(self.s2_head(d2)),
                    up(self.s3_head(d3)),
                    up(self.s4_head(d4)),
                ],
            }
        return main_logit


class ExportWrapper(nn.Module):
    """Wraps model for ONNX export: adds sigmoid to raw logits."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x)
        if isinstance(out, dict):
            out = out['main']
        if isinstance(out, (list, tuple)):
            out = out[0]
        return torch.sigmoid(out)


# ── Instantiate ──
model = EfficientUNetPP().to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\u2713 EfficientUNetPP created')
print(f'  Total params:     {total_params:,}')
print(f'  Trainable params: {trainable:,}')
print(f'  Size (FP32):      {total_params * 4 / 1024 / 1024:.1f} MB')

# ── Validate ──
model.train()
dummy = torch.randn(1, 3, 192, 192, device=device)
out = model(dummy)
assert isinstance(out, dict), 'Training mode should return dict'
assert out['main'].shape == (1, 1, 192, 192), f'Main shape: {out["main"].shape}'
model.eval()
with torch.no_grad():
    out = model(dummy)
assert out.shape == (1, 1, 192, 192), f'Eval shape: {out.shape}'
print('\u2713 Forward pass validated')
del dummy, out
torch.cuda.empty_cache()

## CELL 6 — Loss Functions

In [ ]:
# ================================================================
# CELL 6 — Loss Functions (ALL receive RAW LOGITS)
# ================================================================

def get_pred(out):
    """Extract main prediction from model output."""
    if isinstance(out, dict):
        return out['main']
    if isinstance(out, (list, tuple)):
        return out[0]
    return out


class BCELoss(nn.Module):
    def forward(self, logits, target):
        return F.binary_cross_entropy_with_logits(logits, target)


class DiceLoss(nn.Module):
    def forward(self, logits, target):
        pred = torch.sigmoid(logits).clamp(1e-6, 1 - 1e-6)
        inter = (pred * target).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        return (1 - (2 * inter + 1) / (union + 1)).mean()


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()


class IoULoss(nn.Module):
    def forward(self, logits, target):
        pred = torch.sigmoid(logits).clamp(1e-6, 1 - 1e-6)
        inter = (pred * target).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) - inter
        return (1 - (inter + 1) / (union + 1)).mean()


class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, logits, target):
        pred = torch.sigmoid(logits).clamp(1e-6, 1 - 1e-6)
        TP = (pred * target).sum(dim=(2, 3))
        FP = (pred * (1 - target)).sum(dim=(2, 3))
        FN = ((1 - pred) * target).sum(dim=(2, 3))
        return (1 - (TP + 1) / (TP + self.alpha * FP + self.beta * FN + 1)).mean()


class BoundaryLoss(nn.Module):
    def forward(self, logits, target):
        dilated = F.max_pool2d(target, 5, 1, 2)
        eroded = -F.max_pool2d(-target, 5, 1, 2)
        boundary = (dilated - eroded).clamp(0, 1)
        weight = 1 + 5 * boundary
        return F.binary_cross_entropy_with_logits(logits, target, weight=weight)


class LovaszLoss(nn.Module):
    """Lovasz-Softmax loss for binary segmentation (optimizes IoU directly)."""
    def forward(self, logits, target):
        pred = torch.sigmoid(logits)
        signs = 2 * target - 1
        errors = 1 - pred * signs
        errors_sorted, perm = torch.sort(errors.view(-1), descending=True)
        gt_sorted = target.view(-1)[perm]
        grad = self._lovasz_grad(gt_sorted)
        return F.relu(errors_sorted).dot(grad)

    @staticmethod
    def _lovasz_grad(gt_sorted):
        p = len(gt_sorted)
        gts = gt_sorted.sum()
        intersection = gts - gt_sorted.cumsum(0)
        union = gts + (1 - gt_sorted).cumsum(0)
        jaccard = 1.0 - intersection / union
        if p > 1:
            jaccard[1:] = jaccard[1:] - jaccard[:-1]
        return jaccard


class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = BCELoss()
        self.dice = DiceLoss()
        self.focal = FocalLoss()
        self.iou = IoULoss()
        self.tversky = TverskyLoss()
        self.lovasz = LovaszLoss()
        self.boundary = BoundaryLoss()

    def forward(self, outputs, target):
        H, W = target.shape[2:]

        def resize(t):
            if t.shape[2:] != (H, W):
                return F.interpolate(t, size=(H, W), mode='bilinear', align_corners=False)
            return t

        if isinstance(outputs, dict):
            main = resize(outputs['main'])
            task_loss = (
                0.20 * self.bce(main, target)
                + 0.25 * self.dice(main, target)
                + 0.15 * self.focal(main, target)
                + 0.10 * self.iou(main, target)
                + 0.15 * self.tversky(main, target)
                + 0.15 * self.lovasz(main, target)
            )
            total = task_loss

            if 'boundary' in outputs:
                boundary_logit = resize(outputs['boundary'])
                total = total + 0.3 * self.boundary(boundary_logit, target)

            if 'deep' in outputs:
                deep_losses = []
                for s in outputs['deep']:
                    s = resize(s)
                    deep_losses.append(self.dice(s, target))
                total = total + 0.2 * (sum(deep_losses) / len(deep_losses))

            return total
        else:
            logits = resize(outputs)
            return (
                0.20 * self.bce(logits, target)
                + 0.25 * self.dice(logits, target)
                + 0.15 * self.focal(logits, target)
                + 0.10 * self.iou(logits, target)
                + 0.15 * self.tversky(logits, target)
                + 0.15 * self.lovasz(logits, target)
            )


criterion = CombinedLoss().to(device)
print('\u2713 Combined loss function ready (7 components + boundary + deep supervision)')

## CELL 7 — EMA

In [ ]:
# ================================================================
# CELL 7 — Exponential Moving Average
# ================================================================

class EMA:
    """Exponential Moving Average of model parameters.

    ema_weight = decay * ema_weight + (1 - decay) * weight
    Typically gives +0.3-0.5% IoU improvement.
    """
    def __init__(self, model, decay=0.9999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name].mul_(self.decay).add_(
                    param.data, alpha=1 - self.decay
                )

    def apply_shadow(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])

    def restore(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data.copy_(self.backup[name])
        self.backup = {}


ema = EMA(model, CONFIG['ema_decay'])
print('\u2713 EMA initialized (decay={})'.format(CONFIG['ema_decay']))

## CELL 8 — Metrics + Optimizer

In [ ]:
# ================================================================
# CELL 8 — Metrics + Optimizer + Scheduler
# ================================================================

def compute_iou(prob, target, thresh=0.5):
    pb = (prob > thresh).float()
    tb = (target > thresh).float()
    inter = (pb * tb).sum()
    union = pb.sum() + tb.sum() - inter
    return ((inter + 1e-7) / (union + 1e-7)).item()


def compute_dice(prob, target, thresh=0.5):
    pb = (prob > thresh).float()
    tb = (target > thresh).float()
    inter = (pb * tb).sum()
    return ((2 * inter + 1e-7) / (pb.sum() + tb.sum() + 1e-7)).item()


def compute_mae(prob, target):
    return (prob - target).abs().mean().item()


def tta_predict(model, img):
    """Test Time Augmentation: average predictions from 4 views."""
    preds = []
    with torch.no_grad():
        out = model(img)
        preds.append(torch.sigmoid(get_pred(out)))

        flipped_h = torch.flip(img, [3])
        out = model(flipped_h)
        preds.append(torch.sigmoid(get_pred(out)).flip([3]))

        flipped_v = torch.flip(img, [2])
        out = model(flipped_v)
        preds.append(torch.sigmoid(get_pred(out)).flip([2]))

        rotated = torch.rot90(img, 1, [2, 3])
        out = model(rotated)
        preds.append(torch.sigmoid(get_pred(out)).rot90(-1, [2, 3]))

    return torch.stack(preds).mean(0)


# ── Optimizer ──
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
    betas=(0.9, 0.999),
    eps=1e-8,
)

# ── Scheduler ──
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=CONFIG['T_0'],
    T_mult=CONFIG['T_mult'],
    eta_min=CONFIG['eta_min'],
)

# ── AMP Scaler ──
scaler = GradScaler(enabled=torch.cuda.is_available())

print('\u2713 Optimizer: AdamW (lr={}, wd={})'.format(CONFIG['lr'], CONFIG['weight_decay']))
print('\u2713 Scheduler: CosineAnnealingWarmRestarts (T0={}, Tmult={})'.format(CONFIG['T_0'], CONFIG['T_mult']))
print('\u2713 GradScaler: enabled={}'.format(torch.cuda.is_available()))

## CELL 9 — Checkpoint System

In [ ]:
# ================================================================
# CELL 9 — Checkpoint System (Drive-first resume)
# ================================================================

def save_checkpoint(epoch, best_iou, val_iou, phase, is_best=False):
    ckpt = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'ema_state': ema.shadow,
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'best_iou': best_iou,
        'val_iou': val_iou,
        'phase': phase,
        'config': CONFIG,
    }

    local_path = f"{CONFIG['local_ckpt_dir']}/last_checkpoint.pth"
    torch.save(ckpt, local_path)

    if is_best:
        best_local = f"{CONFIG['local_ckpt_dir']}/best_model.pth"
        torch.save(ckpt, best_local)

    if DRIVE_MOUNTED:
        try:
            drive_path = f'{DRIVE_DIR}/checkpoints/last_checkpoint.pth'
            shutil.copy(local_path, drive_path)
            if is_best:
                shutil.copy(best_local, f'{DRIVE_DIR}/checkpoints/best_model.pth')
        except Exception as e:
            print(f'  Drive save warning: {e}')


def load_checkpoint():
    """Try Drive first, then local. Returns (start_epoch, best_iou, phase) or (0, 0, 1)."""
    candidates = []
    if DRIVE_MOUNTED:
        candidates.append(f'{DRIVE_DIR}/checkpoints/last_checkpoint.pth')
    candidates.append(f"{CONFIG['local_ckpt_dir']}/last_checkpoint.pth")

    for path in candidates:
        if os.path.exists(path):
            print(f'Loading checkpoint: {path}')
            ckpt = torch.load(path, map_location=device, weights_only=False)
            model.load_state_dict(ckpt['model_state'])
            ema.shadow = ckpt['ema_state']
            optimizer.load_state_dict(ckpt['optimizer_state'])
            scheduler.load_state_dict(ckpt['scheduler_state'])
            scaler.load_state_dict(ckpt['scaler_state'])
            start_epoch = ckpt['epoch'] + 1
            best_iou = ckpt['best_iou']
            phase = ckpt.get('phase', 1)
            print(f'\u2713 Resumed from epoch {start_epoch}, best IoU: {best_iou:.4f}, phase: {phase}')
            return start_epoch, best_iou, phase

    print('No checkpoint found. Starting fresh.')
    return 0, 0.0, 1


start_epoch, best_iou, current_phase = load_checkpoint()

## CELL 10 — Training Functions

In [ ]:
# ================================================================
# CELL 10 — Training Functions
# ================================================================

def train_one_epoch(model, ema_obj, loader, optimizer, scaler, criterion, device, accum):
    model.train()
    optimizer.zero_grad()
    total_loss, total_iou, n = 0.0, 0.0, 0

    pbar = tqdm(loader, desc='  Train', leave=False)
    for i, batch in enumerate(pbar):
        imgs = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)

        # Label smoothing
        masks_smooth = masks * CONFIG['label_smooth'] + (1 - CONFIG['label_smooth']) / 2

        # Mixup
        if random.random() < CONFIG['mixup_prob'] and imgs.size(0) > 1:
            lam = np.random.beta(CONFIG['mixup_alpha'], CONFIG['mixup_alpha'])
            idx = torch.randperm(imgs.size(0), device=device)
            imgs = lam * imgs + (1 - lam) * imgs[idx]
            masks_smooth = lam * masks_smooth + (1 - lam) * masks_smooth[idx]

        with autocast():
            outs = model(imgs)
            loss = criterion(outs, masks_smooth) / accum

        # NaN guard
        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad()
            continue

        scaler.scale(loss).backward()

        if (i + 1) % accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            ema_obj.update(model)

        with torch.no_grad():
            pred = get_pred(outs)
            prob = torch.sigmoid(pred)
            total_iou += compute_iou(prob, masks)
            total_loss += loss.item() * accum
            n += 1

        pbar.set_postfix(loss=f'{total_loss / max(n, 1):.4f}', iou=f'{total_iou / max(n, 1):.4f}')

    # Flush remaining gradients
    if n > 0 and (n % accum) != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        ema_obj.update(model)

    return total_loss / max(n, 1), total_iou / max(n, 1)


def validate(model, loader, criterion, device, use_tta=False):
    model.eval()
    t_iou, t_dice, t_mae, n = 0.0, 0.0, 0.0, 0

    with torch.no_grad():
        for batch in tqdm(loader, desc='  Val', leave=False):
            imgs = batch['image'].to(device, non_blocking=True)
            masks = batch['mask'].to(device, non_blocking=True)

            if use_tta:
                prob = tta_predict(model, imgs)
            else:
                outs = model(imgs)
                pred = get_pred(outs)
                prob = torch.sigmoid(pred)

            if prob.shape != masks.shape:
                prob = F.interpolate(prob, size=masks.shape[2:], mode='bilinear', align_corners=False)

            t_iou += compute_iou(prob, masks)
            t_dice += compute_dice(prob, masks)
            t_mae += compute_mae(prob, masks)
            n += 1

    return t_iou / max(n, 1), t_dice / max(n, 1), t_mae / max(n, 1)


def visualize_predictions(model, loader, device, num=4):
    model.eval()
    batch = next(iter(loader))
    imgs = batch['image'][:num].to(device)
    masks = batch['mask'][:num]

    with torch.no_grad():
        outs = model(imgs)
        pred = get_pred(outs)
        probs = torch.sigmoid(pred).cpu()

    fig, axes = plt.subplots(num, 3, figsize=(12, 4 * num))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    for i in range(num):
        img = imgs[i].cpu().numpy().transpose(1, 2, 0)
        img = (img * std + mean).clip(0, 1)
        gt = masks[i, 0].numpy()
        pr = probs[i, 0].numpy()
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Input')
        axes[i, 0].axis('off')
        axes[i, 1].imshow(gt, cmap='gray')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        axes[i, 2].imshow(pr, cmap='gray')
        axes[i, 2].set_title(f'Pred (IoU={compute_iou(torch.tensor(pr).unsqueeze(0).unsqueeze(0), torch.tensor(gt).unsqueeze(0).unsqueeze(0)):.3f})')
        axes[i, 2].axis('off')
    plt.tight_layout()
    plt.show()


print('\u2713 Training functions ready')

## CELL 11 — Run Training

In [ ]:
# ================================================================
# CELL 11 — Run Training (Progressive: 192 → 256 → 320)
# ================================================================

EPOCHS = CONFIG['epochs']
ACCUM = CONFIG['accum_steps']
PATIENCE = CONFIG['patience']

no_improve = 0
history = {'train_loss': [], 'train_iou': [], 'val_iou': [], 'val_dice': [], 'val_mae': [], 'lr': []}

# ── Rebuild loaders if resuming into a later phase ──
if current_phase == 2:
    print('Rebuilding loaders for Phase 2 (256)...')
    train_loader, val_loader, test_loader = build_loaders(CONFIG['phase2_size'])
elif current_phase == 3:
    print('Rebuilding loaders for Phase 3 (320)...')
    train_loader, val_loader, test_loader = build_loaders(CONFIG['phase3_size'])

print(f'\n{"="*70}')
print(f'TRAINING START | Epochs: {start_epoch+1}→{EPOCHS} | Best IoU: {best_iou:.4f}')
print(f'{"="*70}')

for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()

    # ── Phase transitions ──
    if epoch == CONFIG['phase2_epoch'] and current_phase < 2:
        current_phase = 2
        print(f'\n\u2550\u2550 PHASE 2: size=256, lr={CONFIG["phase2_lr"]:.1e} \u2550\u2550')
        train_loader, val_loader, test_loader = build_loaders(CONFIG['phase2_size'])
        for pg in optimizer.param_groups:
            pg['lr'] = CONFIG['phase2_lr']

    if epoch == CONFIG['phase3_epoch'] and current_phase < 3:
        current_phase = 3
        print(f'\n\u2550\u2550 PHASE 3: size=320, lr={CONFIG["phase3_lr"]:.1e} \u2550\u2550')
        train_loader, val_loader, test_loader = build_loaders(CONFIG['phase3_size'])
        for pg in optimizer.param_groups:
            pg['lr'] = CONFIG['phase3_lr']

    # ── Warmup ──
    if epoch < CONFIG['warmup_epochs']:
        warmup_lr = CONFIG['lr'] * (epoch + 1) / CONFIG['warmup_epochs']
        for pg in optimizer.param_groups:
            pg['lr'] = warmup_lr

    current_lr = optimizer.param_groups[0]['lr']

    # ── Train ──
    train_loss, train_iou = train_one_epoch(
        model, ema, train_loader, optimizer, scaler, criterion, device, ACCUM
    )

    # ── Validate (EMA only after epoch 10) ──
    use_ema_val = (epoch >= 10)
    if use_ema_val:
        ema.apply_shadow(model)
    val_iou, val_dice, val_mae = validate(model, val_loader, criterion, device)
    if use_ema_val:
        ema.restore(model)
    # ── Schedule ──
    if epoch >= CONFIG['warmup_epochs']:
        scheduler.step()

    # ── Log ──
    epoch_time = time.time() - epoch_start
    history['train_loss'].append(train_loss)
    history['train_iou'].append(train_iou)
    history['val_iou'].append(val_iou)
    history['val_dice'].append(val_dice)
    history['val_mae'].append(val_mae)
    history['lr'].append(current_lr)

    is_best = val_iou > best_iou
    if is_best:
        best_iou = val_iou
        no_improve = 0
        marker = ' \u2605 NEW BEST'
    else:
        no_improve += 1
        marker = f' ({no_improve}/{PATIENCE})'

    print(
        f'Ep {epoch+1:3d}/{EPOCHS} P{current_phase} | '
        f'Loss: {train_loss:.4f} | '
        f'Train IoU: {train_iou:.4f} | '
        f'Val IoU: {val_iou:.4f} | '
        f'Dice: {val_dice:.4f} | '
        f'MAE: {val_mae:.4f} | '
        f'LR: {current_lr:.1e} | '
        f'{epoch_time:.0f}s{marker}'
    )

    # ── Save ──
    save_checkpoint(epoch, best_iou, val_iou, current_phase, is_best)

    # ── Visualize every 10 epochs ──
    if (epoch + 1) % 10 == 0:
        ema.apply_shadow(model)
        visualize_predictions(model, val_loader, device)
        ema.restore(model)

    # ── Early stopping ──
    if no_improve >= PATIENCE:
        print(f'\n\u26a0 Early stopping at epoch {epoch+1}')
        break

    torch.cuda.empty_cache()
    gc.collect()

print(f'\n{"="*70}')
print(f'TRAINING COMPLETE \u2014 Best Val IoU: {best_iou:.4f}')
print(f'{"="*70}')

# ── Plot curves ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep_range, history['train_loss'], label='Train Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_range, history['train_iou'], label='Train IoU', linewidth=2)
axes[1].plot(ep_range, history['val_iou'], label='Val IoU', linewidth=2)
axes[1].axhline(y=0.95, color='green', linestyle='--', alpha=0.5, label='Target (0.95)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU')
axes[1].set_title('IoU Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(ep_range, history['lr'], label='LR', linewidth=2, color='orange')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## CELL 12 — Export + Download

In [ ]:
# ================================================================
# CELL 12 — Export + Download (SELF-CONTAINED)
# ================================================================
# This cell is fully self-contained: it redefines ALL model classes
# so it works even if earlier cells were lost to a Colab restart.
# ================================================================

import os, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Architecture (self-contained) ──

class _HSwish(nn.Module):
    def forward(self, x):
        return x * F.relu6(x + 3.0, inplace=False) / 6.0

class _SEBlock(nn.Module):
    def __init__(self, ch, r=4):
        super().__init__()
        mid = max(ch // r, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(ch, mid, 1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, ch, 1, bias=True),
        )
    def forward(self, x):
        s = self.pool(x)
        s = self.fc(s)
        s = F.relu6(s + 3.0, inplace=False) / 6.0
        return x * s

class _MBConv(nn.Module):
    def __init__(self, inp, oup, stride, expand, kernel=3, use_se=False):
        super().__init__()
        self.use_residual = (stride == 1 and inp == oup)
        mid = int(round(inp * expand))
        layers = []
        if expand != 1:
            layers += [nn.Conv2d(inp, mid, 1, bias=False), nn.BatchNorm2d(mid), _HSwish()]
        layers += [
            nn.Conv2d(mid, mid, kernel, stride=stride, padding=kernel // 2, groups=mid, bias=False),
            nn.BatchNorm2d(mid), _HSwish(),
        ]
        if use_se:
            layers.append(_SEBlock(mid))
        layers += [nn.Conv2d(mid, oup, 1, bias=False), nn.BatchNorm2d(oup)]
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        out = self.block(x)
        return out + x if self.use_residual else out

class _ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 4)
        self.mlp = nn.Sequential(nn.Linear(ch, mid, bias=False), nn.ReLU(inplace=True), nn.Linear(mid, ch, bias=False))
    def forward(self, x):
        B, C, H, W = x.shape
        gap = x.mean(dim=(2, 3))
        gmp = x.amax(dim=(2, 3))
        gate = torch.sigmoid(self.mlp(gap) + self.mlp(gmp))
        return x * gate.view(B, C, 1, 1)

class _SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx = x.amax(dim=1, keepdim=True)
        gate = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * gate

class _CBAM(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.ca = _ChannelAttention(ch, r)
        self.sa = _SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))

class _DecoderNode(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        total_in = in_ch + skip_ch
        self.conv = nn.Sequential(
            nn.Conv2d(total_in, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(),
        )
        self.cbam = _CBAM(out_ch)
        self.drop = nn.Dropout2d(0.1)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        x = self.cbam(x)
        x = self.drop(x)
        return x

class _BoundaryHead(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        mid = max(in_ch // 2, 8)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, mid, 3, padding=1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, 1, 1),
        )
    def forward(self, x):
        return self.conv(x)

class _EfficientUNetPP(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 48, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(48), _HSwish())
        self.e1 = nn.Sequential(_MBConv(48, 24, stride=2, expand=1, kernel=3), _MBConv(24, 24, stride=1, expand=1, kernel=3))
        self.e2 = nn.Sequential(_MBConv(24, 32, stride=2, expand=6, kernel=3), _MBConv(32, 32, stride=1, expand=6, kernel=3))
        self.e3 = nn.Sequential(_MBConv(32, 56, stride=2, expand=6, kernel=5, use_se=True), _MBConv(56, 56, stride=1, expand=6, kernel=5, use_se=True), _MBConv(56, 56, stride=1, expand=6, kernel=5, use_se=True))
        self.e4 = nn.Sequential(_MBConv(56, 112, stride=2, expand=6, kernel=3, use_se=True), _MBConv(112, 112, stride=1, expand=6, kernel=3, use_se=True), _MBConv(112, 112, stride=1, expand=6, kernel=3, use_se=True))
        self.e5 = nn.Sequential(nn.Conv2d(112, 672, 1, bias=False), nn.BatchNorm2d(672), _HSwish())
        self.d1 = _DecoderNode(672, 112, 256)
        self.d2 = _DecoderNode(256, 56, 128)
        self.d3 = _DecoderNode(128, 32, 64)
        self.d4 = _DecoderNode(64, 24, 32)
        self.d5 = nn.Sequential(nn.Conv2d(32, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.GELU(), nn.Conv2d(16, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.GELU())
        self.out_head = nn.Conv2d(16, 1, 1)
        self.s2_head = nn.Conv2d(128, 1, 1)
        self.s3_head = nn.Conv2d(64, 1, 1)
        self.s4_head = nn.Conv2d(32, 1, 1)
        self.boundary_head = _BoundaryHead(17)
    def forward(self, x):
        H, W = x.shape[2:]
        s = self.stem(x); e1 = self.e1(s); e2 = self.e2(e1); e3 = self.e3(e2); e4 = self.e4(e3); e5 = self.e5(e4)
        d1 = self.d1(e5, e4); d2 = self.d2(d1, e3); d3 = self.d3(d2, e2); d4 = self.d4(d3, e1)
        d5 = self.d5(F.interpolate(d4, size=(H, W), mode='bilinear', align_corners=False))
        return self.out_head(d5)

class _ExportWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        out = self.model(x)
        if isinstance(out, dict): out = out['main']
        if isinstance(out, (list, tuple)): out = out[0]
        return torch.sigmoid(out)


# ════════════════════════════════════════════════════════
# EXPORT PIPELINE
# ════════════════════════════════════════════════════════

DRIVE_DIR = '/content/drive/MyDrive/bg_removal_ai'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Step 1: Load best checkpoint ──
ckpt_paths = [
    f'{DRIVE_DIR}/checkpoints/best_model.pth',
    '/content/checkpoints/best_model.pth',
    f'{DRIVE_DIR}/checkpoints/last_checkpoint.pth',
    '/content/checkpoints/last_checkpoint.pth',
]

ckpt_path = None
for p in ckpt_paths:
    if os.path.exists(p):
        ckpt_path = p
        break

assert ckpt_path is not None, 'No checkpoint found! Train first.'
print(f'Loading checkpoint: {ckpt_path}')

ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
print(f'  Epoch: {ckpt["epoch"]}, Best IoU: {ckpt["best_iou"]:.4f}')

# ── Step 2: Build model + load EMA weights ──
export_model = _EfficientUNetPP()

if 'ema_state' in ckpt and ckpt['ema_state']:
    print('  Using EMA weights for export')
    state = export_model.state_dict()
    for name, param in ckpt['ema_state'].items():
        if name in state:
            state[name].copy_(param)
    export_model.load_state_dict(state)
else:
    print('  Using regular weights for export')
    export_model.load_state_dict(ckpt['model_state'])

export_model.eval()
export_model.cpu()

# ── Step 3: Wrap for export (adds sigmoid) ──
wrapper = _ExportWrapper(export_model)
wrapper.eval()

# ── Step 4: Export to ONNX ──
dummy = torch.randn(1, 3, 320, 320)
raw_path = '/content/model_raw.onnx'
final_path = '/content/model_FINAL.onnx'

print('\nExporting to ONNX...')
torch.onnx.export(
    wrapper, dummy, raw_path,
    opset_version=13,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch', 2: 'height', 3: 'width'},
        'output': {0: 'batch', 2: 'height', 3: 'width'},
    },
)
print(f'  Raw export: {os.path.getsize(raw_path)/1024/1024:.1f} MB')

# ── Step 5: Merge into SINGLE file (CRITICAL) ──
print('Merging into single file...')
import onnx
model_proto = onnx.load(raw_path, load_external_data=True)
onnx.save(model_proto, final_path, save_as_external_data=False)
final_size = os.path.getsize(final_path) / 1024 / 1024
print(f'  model_FINAL.onnx: {final_size:.1f} MB (SINGLE FILE \u2713)')

# ── Step 6: Verify with ONNX Runtime ──
print('\nVerifying with ONNX Runtime...')
import onnxruntime as ort
sess = ort.InferenceSession(final_path, providers=['CPUExecutionProvider'])
inp_name = sess.get_inputs()[0].name
test_input = np.random.randn(1, 3, 320, 320).astype(np.float32)
result = sess.run(None, {inp_name: test_input})
assert result[0].shape == (1, 1, 320, 320), f'Bad shape: {result[0].shape}'
assert 0 <= result[0].min() <= result[0].max() <= 1, 'Output not in [0,1]'
print(f'  Output shape: {result[0].shape} \u2713')
print(f'  Output range: [{result[0].min():.4f}, {result[0].max():.4f}] \u2713')
print('  VERIFIED \u2705')

# ── Step 7: Copy to Drive ──
if os.path.isdir(DRIVE_DIR):
    drive_export = f'{DRIVE_DIR}/exports/model_FINAL.onnx'
    shutil.copy(final_path, drive_export)
    print(f'\n\u2713 Copied to Drive: {drive_export}')

# Clean up raw
if os.path.exists(raw_path):
    os.remove(raw_path)

# ── Step 8: Auto-download ──
print(f'\n{"="*70}')
print(f'model_FINAL.onnx = {final_size:.1f} MB')
print(f'Best IoU: {ckpt["best_iou"]:.4f}')
print(f'{"="*70}')
print('\nDownloading model_FINAL.onnx...')

try:
    from google.colab import files
    files.download(final_path)
except Exception:
    print(f'Auto-download failed. Download manually from: {final_path}')
    if os.path.isdir(DRIVE_DIR):
        print(f'Or from Drive: {DRIVE_DIR}/exports/model_FINAL.onnx')